In [12]:
from freqtrade_client import FtRestClient
server_url = 'http://127.0.0.1:8080'
username = ''
password = ""
client = FtRestClient(server_url, username, password)

In [39]:
client.status()[0]['profit_pct']

0.64

In [83]:
starting_balance = client.daily(1).get('data')[0].get('starting_balance')
dataframe = pd.DataFrame(client.trades().get('trades'))
dataframe['rel_stake'] = dataframe['stake_amount'] / starting_balance
dataframe['rel_profit'] = dataframe['close_profit_pct'] * dataframe['rel_stake']
rel_profit = dataframe[
    (dataframe.close_date > (date.today() - timedelta(days=1)).strftime('%Y-%m-%d')) &
    (dataframe.close_profit_pct > 0)
].rel_profit.sum()
rel_profit

8.591147258713216

In [6]:
stoploss = -0.02
max_stake = 300
min_stake = 5
risk = 0.3
max(min(max_stake * abs(stoploss) / risk, max_stake), min_stake)

20.0

In [1]:
import ccxt
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import datetime

In [2]:
def dataframe_from_csv(columns):
    dataframe = pd.read_csv("out.csv")
    dataframe = dataframe[columns]
    dataframe['date'] = pd.to_datetime(dataframe['date'])
    return dataframe

In [12]:
bybit = ccxt.bybit()

In [3]:
def return_order_book(symbol='BTC/USDT', n=200):
    bybit_ob = bybit.fetchOrderBook(symbol, n)
    binance_ob = bybit.fetchOrderBook(symbol, n)
    kucoin_ob = bybit.fetchOrderBook(symbol, n)
    bid_values = {
        'price': np.hstack((np.array(bybit_ob['bids'])[:,0], np.array(binance_ob['bids'])[:,0], np.array(kucoin_ob['bids'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['bids'])[:,1], np.array(binance_ob['bids'])[:,1], np.array(kucoin_ob['bids'])[:,1])),
        'side':'bid'
    }
    ask_values = {
        'price': np.hstack((np.array(bybit_ob['asks'])[:,0], np.array(binance_ob['asks'])[:,0], np.array(kucoin_ob['asks'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['asks'])[:,1], np.array(binance_ob['asks'])[:,1], np.array(kucoin_ob['asks'])[:,1])),
        'side':'ask'
    }
    bid_dataframe = pd.DataFrame(bid_values)
    ask_dataframe = pd.DataFrame(ask_values)
    dataframe = pd.concat((bid_dataframe,ask_dataframe))
    dataframe = dataframe.groupby(['price','side']).sum().reset_index()
    # dataframe.groupby('side').sum()
    dataframe['now'] = datetime.datetime.now()
    return dataframe

In [13]:
def fetch_ohlcv(symbol='BTC/USDT', limit=240):
    values = bybit.fetch_ohlcv(symbol=symbol,timeframe='1m', limit=limit)
    dataframe = pd.DataFrame(values, columns=('date','open','high','low','close','volume'))
    dataframe['date'] = pd.to_datetime(dataframe['date'],unit='ms')
    return dataframe

In [20]:
def caculate_regression(dataframe, kernel=1440):
    dataframe_ = dataframe.copy()[-kernel:]
    x = dataframe_.index.values.reshape(-1, 1)
    y = dataframe_.close.values
    model = LinearRegression()
    model.fit(x, y)
    x = dataframe.index.values.reshape(-1, 1)
    dataframe['y'] = model.predict(x)
    dataframe['coef'] = float(model.coef_[0])
    dataframe['upper_band'] = dataframe['y'] + dataframe.close.std()
    dataframe['lower_band'] = dataframe['y'] - dataframe.close.std()
    dataframe['band_dist'] = dataframe['upper_band'] - dataframe['lower_band']
    return dataframe

In [4]:
def calculate_extrema(dataframe, kernel=12):
    dataframe["extrema"] = 0
    min_peaks = argrelextrema(dataframe["low"].values, np.less_equal, order=kernel)
    max_peaks = argrelextrema(dataframe["high"].values, np.greater_equal, order=kernel)
    for mp in min_peaks[0]:
        dataframe.at[mp, "extrema"] = -1
    for mp in max_peaks[0]:
        dataframe.at[mp, "extrema"] = 1
    dataframe['last_min_peak'] = dataframe.at[min_peaks[0][-1], "low"]
    dataframe['last_max_peak'] = dataframe.at[max_peaks[0][-1], "high"]
    dataframe['h_dist'] = np.where(dataframe.extrema == 1, (dataframe.high - dataframe.upper_band), 0)
    dataframe['l_dist'] = np.where(dataframe.extrema == -1, (dataframe.lower_band - dataframe.low), 0)
    dataframe['h_ratio'] = dataframe['h_dist'] / dataframe['band_dist']
    dataframe['l_ratio'] = dataframe['l_dist'] / dataframe['band_dist']
    dataframe['l_h_ratio'] = dataframe.at[max_peaks[0][-1], "h_ratio"]
    dataframe['l_l_ratio'] = dataframe.at[min_peaks[0][-1], "l_ratio"]
    return dataframe

In [16]:
def plot(symbol='BTC/USDT', kernel=24):

    columns = ['date','open','high','low','close','volume']
    dataframe = dataframe_from_csv(columns)

    fig = go.Figure(data=[go.Candlestick(x=dataframe.index.values,
                    open=dataframe['open'],
                    high=dataframe['high'],
                    low=dataframe['low'],
                    close=dataframe['close'],
                    increasing_line_color= 'green', 
                    decreasing_line_color= 'red')])

    dataframe = caculate_regression(dataframe, kernel=1440)

    fig.add_scatter(
        x= dataframe.index.values, 
        y= dataframe.upper_band.values,
        mode="lines", 
        marker=dict(size=7, color="green")
    )

    fig.add_scatter(
        x= dataframe.index.values, 
        y= dataframe.y.values,
        mode="lines", 
        marker=dict(size=7, color="blue")
    )

    fig.add_scatter(
        x= dataframe.index.values, 
        y= dataframe.lower_band.values,
        mode="lines", 
        marker=dict(size=7, color="red")
    )

    df = calculate_extrema(dataframe, kernel=kernel)

    fig.add_scatter(
        x= df[df.extrema == 1].index.values, 
        y= df[df.extrema == 1].high.values,
        mode="markers", 
        marker=dict(size=7, color="purple")
    )

    fig.add_scatter(
        x= df[df.extrema == -1].index.values, 
        y= df[df.extrema == -1].low.values,
        mode="markers", 
        marker=dict(size=7, color="yellow")
    )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    fig.update_layout(autosize=True, height=500,xaxis_rangeslider_visible=False)
    fig.show()

In [6]:
columns = ['date','open','high','low','close','volume']
dataframe = dataframe_from_csv(columns)
# dataframe = caculate_regression(dataframe, kernel=240)
# dataframe = calculate_extrema(dataframe, kernel=6)
# columns = ['high','low','coef','y','lower_band','upper_band','close','l_h_ratio','l_l_ratio']
dataframe[columns].tail(5)

,date,open,high,low,close,volume
1235,2024-11-16 14:07:00+00:00,3.7521,3.7521,3.7417,3.7452,36638.0
1236,2024-11-16 14:08:00+00:00,3.7452,3.7497,3.7434,3.7467,20214.0
1237,2024-11-16 14:09:00+00:00,3.7467,3.7494,3.7418,3.7439,16005.0
1238,2024-11-16 14:10:00+00:00,3.7439,3.7439,3.7343,3.7343,37831.0
1239,2024-11-16 14:11:00+00:00,3.7343,3.7372,3.7307,3.7333,57004.0


In [25]:
plot(symbol='BTC/USDT', kernel=24)